# <b><u>Note to reader</u></b>

This notebook contains little tasks done at various time : during WCS training, for or during the certification, or as little personal project.
<p>------------------------</p>

# A - Géocoding ou géocodage - API Nomanatim
Le géocodage est le processus de transformation d'une description d'emplacement, telle qu'une adresse postale, en coordonnées géographiques (latitude et longitude). Cela permet de représenter l'emplacement sur une carte et de l'utiliser pour diverses applications, telles que la navigation, la visualisation de données géographiques et l'analyse spatiale.

### Basé sur l'adresse

Géocodage basé sur l'adresse (Address Geocoding) : C'est la forme la plus courante de géocodage. Elle consiste à prendre une adresse postale, telle qu'un numéro de rue, un nom de rue, une ville et un pays, et à la convertir en coordonnées géographiques. Les services de géocodage utilisent généralement des bases de données d'adresses pour associer l'adresse fournie à des coordonnées géographiques précises.

### Basé sur les coordonnées géographiques
Géocodage inversé (Reverse Geocoding) : Contrairement au géocodage basé sur l'adresse, le géocodage inversé prend des coordonnées géographiques (latitude et longitude) et les convertit en une adresse postale ou une description d'emplacement. Cela peut être utile lorsque vous avez uniquement les coordonnées d'un point et que vous souhaitez connaître l'adresse associée.

### Basé sur des POI
Géocodage basé sur des points d'intérêt (POI Geocoding) : Ce type de géocodage permet de localiser des points d'intérêt spécifiques, tels que des restaurants, des hôtels, des stations-service, etc. Il utilise une base de données de points d'intérêt pour associer le nom du lieu à des coordonnées géographiques.

## Scraping 1 - Meeting point

In [ ]:
# Doc API a voir ici
https://nominatim.org/release-docs/latest/api/Search/

Vous êtes des employés remote, quel serait le meilleur lieu de rdv pour tout le monde ?

### <b>Etape 1 : Récupérer les adresses</b>

In [1]:
import pandas as pd
import requests
import json
import pprint
import folium

In [2]:
#dataframe

df = pd.DataFrame({"employé": ["emma", "noah", "sophia", "liam", "olivia", "william", "ava", "james", "isabella", "benjamin"],
                           "ville": ["paris", "marseille", "lyon", "toulouse", "nice", "nantes", "strasbourg", "montpellier", "bordeaux", "lille"]})
df

,employé,ville
0,emma,paris
1,noah,marseille
2,sophia,lyon
3,liam,toulouse
4,olivia,nice
5,william,nantes
6,ava,strasbourg
7,james,montpellier
8,isabella,bordeaux
9,benjamin,lille


In [3]:
#Mettre 1eres lettres des employés et villes en majuscule

df = df.apply(lambda x: x.str.capitalize())
df

,employé,ville
0,Emma,Paris
1,Noah,Marseille
2,Sophia,Lyon
3,Liam,Toulouse
4,Olivia,Nice
5,William,Nantes
6,Ava,Strasbourg
7,James,Montpellier
8,Isabella,Bordeaux
9,Benjamin,Lille


### <b>Etape 2 : Récupérer lat long sur Nomanatim</b>

Ne pas runner inutilement les cellules pour moins de call API

In [4]:
# Définir une fonction qui retourne les coordonnées GPS d'une ville donnée en argument:

def get_coordinates(town):
    link_main = "https://nominatim.openstreetmap.org/search?q="
    link_end = '&format=json&limit=1'
    search_link = link_main + town + link_end
    navigator = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/100.0.4896.60 Safari/537.36'
    response = requests.get(search_link, headers={'User-Agent': navigator})
    response_json = response.json()
    if response.status_code != 200:
        print("Error:", response.status_code)
        return None
    if not response_json:
        print("No results found")
        return None
    lat = response_json[0]['lat']
    lon = response_json[0]['lon']
    return float(lat), float(lon)

In [5]:
get_coordinates('Annemasse')

(46.1934005, 6.2341093)

In [6]:
get_coordinates('Annemasse')[0] # latitude

46.1934005

In [7]:
# Définir une fonction qui retourne les coordonnées a partir de l'adresse complete:
def get_GPS_adresse(adresse_postale):
  link_main = "https://nominatim.openstreetmap.org/search?q="
  link_end = '&format=json&limit=1'
  search_link = link_main + adresse_postale.replace(' ', '+') + link_end
  headers =  {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:100.0) Gecko/20100101 Firefox/100.0',
    'Referer': 'https://www.example.com'}
  response = requests.get(search_link, headers=headers)
  response_json = response.json()
  coordinates = "(" + response_json[0]['lat'] + ", " + response_json[0]['lon'] + ")"
  return coordinates

In [8]:
get_GPS_adresse("7 Rue de Bellevue, 74100 Annemasse")

'(46.1913280, 6.2267070)'

### <b>Etape 3 : Appliqué aux employés</b>

In [ ]:
# Créer une colonne avec les coordonnées des villes des employés en utilisant la fonction get_coordinates et la méthode apply de pandas
# puis Créer deux colonnes latitude et longitude pour les coordonnees - en utilisant la fonction get_coordinates

In [9]:
# Call API ONCE
df['coordonnees'] = df['ville'].apply(get_coordinates)

In [10]:
# Expand tuple into two columns
df[['latitude', 'longitude']] = pd.DataFrame(
    df['coordonnees'].tolist(),
    index=df.index
  )

In [11]:
df

,employé,ville,coordonnees,latitude,longitude
0,Emma,Paris,"(48.8534951, 2.3483915)",48.853495,2.348391
1,Noah,Marseille,"(43.2961743, 5.3699525)",43.296174,5.369953
2,Sophia,Lyon,"(45.7578137, 4.8320114)",45.757814,4.832011
3,Liam,Toulouse,"(43.6044638, 1.4442433)",43.604464,1.444243
4,Olivia,Nice,"(43.7009358, 7.2683912)",43.700936,7.268391
5,William,Nantes,"(47.2186371, -1.5541362)",47.218637,-1.554136
6,Ava,Strasbourg,"(48.584614, 7.7507127)",48.584614,7.750713
7,James,Montpellier,"(43.6112422, 3.8767337)",43.611242,3.876734
8,Isabella,Bordeaux,"(44.841225, -0.5800364)",44.841225,-0.580036
9,Benjamin,Lille,"(50.6365654, 3.0635282)",50.636565,3.063528


### <b>Etape 4 : Pt de rencontre</b>

#### Lieu de rencontre

In [12]:
# Définir un point rencontre en utilisant les moyennes des latitude et longitude (Faites un round 6)

meeting_pt_lat = df['latitude'].mean().round(6)
meeting_pt_long = df['longitude'].mean().round(6)
meeting_pt = (meeting_pt_lat, meeting_pt_long)
meeting_pt

(46.010517, 3.381979)

#### Région de rencontre

In [ ]:
# Définir la region_rencontre
# On utilise la fonction reverse de Nominatim pour trouver la région correspondant au point de rencontre
# On va donc rechercher dans quelle region la latitude et la longitude du point de rencontre se trouvent.

In [13]:
# Preparing the link
reverse_search = "https://nominatim.openstreetmap.org/reverse?"
meeting_pt_link = "lat=" + str(meeting_pt_lat) + "&lon=" + str(meeting_pt_long) 

reverse_search_link = reverse_search + meeting_pt_link + "&format=json"
reverse_search_link

'https://nominatim.openstreetmap.org/reverse?lat=46.010517&lon=3.381979&format=json'

In [14]:
# Requesting the API afin de voir ou se situe la region de rencontre dans le JSON
navigator = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
response = requests.get(reverse_search_link, headers={'User-Agent': navigator})

response_json = response.json()

#pprint.pprint(response_json, compact = True)

In [15]:
response_json

{'place_id': 426551578,
 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright',
 'osm_type': 'node',
 'osm_id': 13418697923,
 'lat': '46.0106686',
 'lon': '3.3817652',
 'class': 'place',
 'type': 'house',
 'place_rank': 30,
 'importance': 6.212329289687979e-05,
 'addresstype': 'place',
 'name': '',
 'display_name': '6, Route de Loriaval, Loriaval, Randan, Riom, Puy-de-Dôme, Auvergne-Rhône-Alpes, France métropolitaine, 63310, France',
 'address': {'house_number': '6',
  'road': 'Route de Loriaval',
  'hamlet': 'Loriaval',
  'village': 'Randan',
  'municipality': 'Riom',
  'county': 'Puy-de-Dôme',
  'ISO3166-2-lvl6': 'FR-63',
  'state': 'Auvergne-Rhône-Alpes',
  'ISO3166-2-lvl4': 'FR-ARA',
  'region': 'France métropolitaine',
  'postcode': '63310',
  'country': 'France',
  'country_code': 'fr'},
 'boundingbox': ['46.0106186', '46.0107186', '3.3817152', '3.3818152']}

In [16]:
# Saving the response into variables
meeting_pt_region = response_json['address']['state']
meeting_pt_dept = response_json['address']['county']

In [17]:
# We can see that the region is named 'state' in the API response and departement is called 'county'. It can be interesting to extract both information.

print("Le lieu de recontre aura lieu en "  + meeting_pt_region + ", idealement dans le departement du " + meeting_pt_dept )


Le lieu de recontre aura lieu en Auvergne-Rhône-Alpes, idealement dans le departement du Puy-de-Dôme


### <b>Etape 5 : Créer une carte</b>

In [18]:
df

,employé,ville,coordonnees,latitude,longitude
0,Emma,Paris,"(48.8534951, 2.3483915)",48.853495,2.348391
1,Noah,Marseille,"(43.2961743, 5.3699525)",43.296174,5.369953
2,Sophia,Lyon,"(45.7578137, 4.8320114)",45.757814,4.832011
3,Liam,Toulouse,"(43.6044638, 1.4442433)",43.604464,1.444243
4,Olivia,Nice,"(43.7009358, 7.2683912)",43.700936,7.268391
5,William,Nantes,"(47.2186371, -1.5541362)",47.218637,-1.554136
6,Ava,Strasbourg,"(48.584614, 7.7507127)",48.584614,7.750713
7,James,Montpellier,"(43.6112422, 3.8767337)",43.611242,3.876734
8,Isabella,Bordeaux,"(44.841225, -0.5800364)",44.841225,-0.580036
9,Benjamin,Lille,"(50.6365654, 3.0635282)",50.636565,3.063528


#### Bounding box / Boite englobante

In [23]:
# Creer une carte avec le cadre Français
map_fr = folium.Map()
map_fr.fit_bounds(
    bounds =[
            [42.96, -2.45], # Sud-Ouest
            [51.1, 8.028] # Nord-Est
    ]
)

#map_fr

In [25]:
# Trying to apply it to our dataframe:
# Ajouter les élèves
# Faites un boucle pour associer une localisation à chaque ligne employé

for i, row in df.iterrows():
    folium.Marker(location=[row["latitude"], row["longitude"]], popup=row["employé"]).add_to(map_fr)
                                                    
map_fr

In [32]:
# Ajout le pt de rencontre - marker + icone sympa
icon_perso = folium.Icon(color ="red", icon="")
popup_perso = folium.Popup(f"Le point de rencontre se trouve dans le " + meeting_pt_dept + " en " + meeting_pt_region)
folium.Marker(
    location = meeting_pt,
    popup = popup_perso,
    icon = icon_perso
).add_to(map_fr)

map_fr

In [ ]:
map_fr.save("map_fr.html")

## Scraping 2 - Markers

<b>Contexte :</b>

Vous êtes un analyste de données travaillant sur un projet pour identifier et visualiser des points d'intérêt (POI) à New York. Votre tâche consiste à récupérer les coordonnées géographiques de ces lieux, à traiter les données JSON obtenues via une API open source (Nominatim), et à afficher ces points sur une carte interactive avec Folium.

### Partie 1 : Récupération des coordonnées avec Nominatim

Objectif : Utiliser l'API Nominatim pour obtenir les coordonnées (latitude et longitude) des points d'intérêt à New York.

In [1]:
import requests
import folium
import re
from typing import Optional
import pandas as pd
import json

In [2]:
url = f"https://nominatim.openstreetmap.org/search?"

In [3]:
locations = [
    'Statue of Liberty, New York, NY',
    'Central Park, New York, NY',
    'Empire State Building, New York, NY',
    'Times Square, New York, NY',
    'Brooklyn Bridge, New York, NY'
]

In [4]:
# Créer la fonction get_lat_lon pour récupérer les coordonnées géographiques d'un emplacement

def get_lat_lon(location: str) -> Optional[tuple]:
    url = f"https://nominatim.openstreetmap.org/search?"
    search_link = url + 'q=' + location + '&format=json&limit=1'
    navigator = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/100.0.4896.60 Safari/537.36'
    response = requests.get(search_link, headers={'User-Agent': navigator})
    response_json = response.json()
    if response.status_code != 200:
        print("Error:", response.status_code)
        return None
    if not response_json:
        print("No results found")
        return None
    lat = response_json[0]['lat']
    lon = response_json[0]['lon']
    return float(lat), float(lon)

In [5]:
get_lat_lon(locations[0])

(40.6892532, -74.0445482)

In [6]:
# Création du dataframe avec les emplacements et les coordonnées géographiques
df = pd.DataFrame({
    "Location": locations,
    "lat_long": [get_lat_lon(loc) for loc in locations]
})

df.head()

,Location,lat_long
0,"Statue of Liberty, New York, NY","(40.6892532, -74.0445482)"
1,"Central Park, New York, NY","(40.7827725, -73.9653627)"
2,"Empire State Building, New York, NY","(40.7484421, -73.9856589)"
3,"Times Square, New York, NY","(40.7572614, -73.9858998)"
4,"Brooklyn Bridge, New York, NY","(40.7062175, -73.9970208)"


### Partie 2 : Afficher les marker sur une carte folium

Objectif : Afficher les coordonnées des lieux touristiques sur une carte folium

In [7]:
# Récupérer les coordonnées de New York à l'aide de la fonction get_lat_lon
new_york_coord = get_lat_lon("New York")
new_york_coord

(40.7127281, -74.0060152)

In [8]:
# Centrer la carte folium sur ces coordonnées
my_map = folium.Map(location= new_york_coord, zoom_start=10)

In [9]:
# Fonction pour placer les marqueurs sur la carte
def add_markers(row, my_map= my_map):
    for i, row in df.iterrows():
        folium.Marker(location= row["lat_long"], popup=row["Location"]).add_to(my_map)
    return my_map

In [10]:
#Ajouter les marqueurs sur la carte
#Afficher la carte folium
add_markers(df)

my_map.save("map.html")

my_map

### Partie 3 : Dealing with Regex

In [11]:
locations_with_descriptions = [
    {'location': 'Statue of Liberty, New York, NY', 'description': 'A famous landmark. Contact us at info@statueofliberty.org or call +1-212-363-3200'},
    {'location': 'Central Park, New York, NY', 'description': 'A large city park. Email: centralpark@nycparks.gov or call +1-212-310-6600'},
    {'location': 'Empire State Building, New York, NY', 'description': 'Iconic skyscraper. For inquiries: contact@empirestatebldg.com or call +1-212-736-3100'},
    {'location': 'Times Square, New York, NY', 'description': 'Major commercial intersection. Reach out at info@timessquare.org or call +1-212-768-1560'},
    {'location': 'Brooklyn Bridge, New York, NY', 'description': 'Historic bridge. Email us at info@brooklynbridge.org or call +1-718-222-9939'}
]


In [12]:
# Créer un dataframe avec les emplacements et les descriptions
df_descriptions = pd.DataFrame(locations_with_descriptions)

In [13]:
df_descriptions

,location,description
0,"Statue of Liberty, New York, NY",A famous landmark. Contact us at info@statueof...
1,"Central Park, New York, NY",A large city park. Email: centralpark@nycparks...
2,"Empire State Building, New York, NY",Iconic skyscraper. For inquiries: contact@empi...
3,"Times Square, New York, NY",Major commercial intersection. Reach out at in...
4,"Brooklyn Bridge, New York, NY",Historic bridge. Email us at info@brooklynbrid...


In [14]:
# Fonction pour extraire l'adresse email de la description

import re

def extract_email(description):
    pattern = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"
    match = re.search(pattern, description)
    if match:
        return match.group(0)
    else:
        return None

In [16]:
#Fonction pour extraire le numéro de téléphone de la description
def extract_phone_number(description):
    pattern = r"\+?\d{1,4}[-.\s]?\(?\d{1,3}?\)?[-.\s]?\d{1,4}[-.\s]?\d{1,4}[-.\s]?\d{1,9}"
    match = re.search(pattern, description)
    if match:
        return match.group(0)
    else:
        return None

In [17]:
# Creer une nouvelle colonne dans le dataframe pour les adresses email extraites
df_descriptions['email'] = df_descriptions['description'].apply(extract_email)

#Ajouter une nouvelle colonne pour les numéros de téléphone extraits au dataframe df_descriptions
df_descriptions["phone_number"] = df_descriptions["description"].apply(extract_phone_number)

df_descriptions.head(2)

,location,description,email,phone_number
0,"Statue of Liberty, New York, NY",A famous landmark. Contact us at info@statueof...,info@statueofliberty.org,+1-212-363-3200
1,"Central Park, New York, NY",A large city park. Email: centralpark@nycparks...,centralpark@nycparks.gov,+1-212-310-6600


In [18]:
# Cleaning the column description from the email and phone number information in the dataframe df_descriptions 
# and from anything after the . (ie no contact us, email us, call, etc in the description)
def clean_description(description):
    description = re.sub(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", "", description)
    description = re.sub(r"\+?\d{1,4}[-.\s]?\(?\d{1,3}?\)?[-.\s]?\d{1,4}[-.\s]?\d{1,4}[-.\s]?\d{1,9}", "", description)
    description = re.sub(r"\..*", "", description)  # Remove anything after the first period
    return description.strip()

In [19]:
# Return the dataset with cleaned descriptions and drop the original description column
df_descriptions["cleaned_description"] = df_descriptions["description"].apply(clean_description)
df_descriptions = df_descriptions.drop(columns=["description"])

In [20]:
df_descriptions

,location,email,phone_number,cleaned_description
0,"Statue of Liberty, New York, NY",info@statueofliberty.org,+1-212-363-3200,A famous landmark
1,"Central Park, New York, NY",centralpark@nycparks.gov,+1-212-310-6600,A large city park
2,"Empire State Building, New York, NY",contact@empirestatebldg.com,+1-212-736-3100,Iconic skyscraper
3,"Times Square, New York, NY",info@timessquare.org,+1-212-768-1560,Major commercial intersection
4,"Brooklyn Bridge, New York, NY",info@brooklynbridge.org,+1-718-222-9939,Historic bridge


### Partie 4 : JSON Manipulation

Objectif : Extraire les lieux touristiques avec plus de 100 000 visiteurs par semaine

In [21]:
json_data = '''
[
    {
        "nom": "Statue of Liberty",
        "description": "A famous landmark in New York.",
        "email": "info@statueofliberty.org",
        "telephone": "+1-212-363-3200",
        "visiteurs_par_jour": 20000
    },
    {
        "nom": "Central Park",
        "description": "A large city park in New York.",
        "email": "centralpark@nycparks.gov",
        "telephone": "+1-212-310-6600",
        "visiteurs_par_jour": 15000
    },
    {
        "nom": "Empire State Building",
        "description": "Iconic skyscraper in New York.",
        "email": "contact@empirestatebldg.com",
        "telephone": "+1-212-736-3100",
        "visiteurs_par_jour": 25000
    },
    {
        "nom": "Times Square",
        "description": "Major commercial intersection in New York.",
        "email": "info@timessquare.org",
        "telephone": "+1-212-768-1560",
        "visiteurs_par_jour": 30000
    },
    {
        "nom": "Brooklyn Bridge",
        "description": "Historic bridge in New York.",
        "email": "info@brooklynbridge.org",
        "telephone": "+1-718-222-9939",
        "visiteurs_par_jour": 10000
    }
]
'''

In [22]:
#Charger le json en tant que dictionnaire -> hint : utiliser une fonction du module json
data = json.loads(json_data)

In [23]:
#Extraire les lieux touristiques avec plus de 100k visiteurs par semaine
lieux_touristiques_100k_visit_semaine = [lieu["nom"] for lieu in data if lieu["visiteurs_par_jour"] > 100000/7]

In [24]:
lieux_touristiques_100k_visit_semaine

['Statue of Liberty', 'Central Park', 'Empire State Building', 'Times Square']

# B - Webscraping with Beautiful Soup

In [44]:
import pandas as pd
import requests
import re
from bs4 import BeautifulSoup

In [45]:
url = "https://www.chucknorrisfacts.fr/facts/top/1"
navigator = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'


html_page = requests.get(url, headers={'User-Agent': navigator})

print(f"Status Code: {html_page.status_code}")

Status Code: 200


In [46]:
# Create a BeautifulSoup object ie parse HTML content
soup = BeautifulSoup(html_page.text, 'html.parser')

In [47]:
# les blagues se trouvent dans une div class "card-body bg-light rounded"
joke = soup.find_all("div", class_="card-body bg-light rounded")

In [48]:
print('Nombre de blagues sur Chuck Norris sur cette page: ', len(joke))

Nombre de blagues sur Chuck Norris sur cette page:  20


#### La 8e blague

In [49]:
# Bloc des blagues
joke_card = soup.find_all("div", class_="card")

In [51]:
# isoler le bloc de la 8e blague 
joke_8th = joke_card[7]

In [52]:
# Prendre uniquement le texte de la 8e blague - dans p.card-text
print(joke_8th.select("p.card-text")[0].get_text())

Chuck Norris ne ment pas, c'est la vérité qui se trompe.


In [53]:
# Extraire la note de la 8e blague
ratings = joke_8th.select('span[id^="moyenne_"]')

# on enlève les parenthèses autour de la note et on affiche la note
print(ratings[0].get_text()[1:-1])

8.33/10


#### Dictionnaire avec note + blague 

In [54]:
#création du dict vide:
jokes_dict = {}

In [55]:
for i in range(len(joke_card)):
    #print(i) 
    try : 
        joke = joke_card[i].select("p.card-text")[0].get_text()
     #print(joke)
    except IndexError:
        pass
    try:
        rating = joke_card[i].select('span[id^="moyenne_"]')[0].get_text()[1:-4] # on enlève les parenthèses autour de la note + le /10
        #print(rating)
    except IndexError:
        pass
    jokes_dict[joke] = rating 

In [56]:
print(jokes_dict)

{"Chuck Norris peut manger plus de donuts qu'Homer.": '8.87', "Les ennemis des amis de Chuck Norris sont ses amis. Et oui! Les ennemis de Chuck Norris n'existent plus.": '8.83', "Un jour Chuck Norris a eu un zero en latin, depuis c'est une langue morte.": '8.42', "L'avenir se demande parfois ce que Chuck Norris lui réserve.": '8.42', 'Chuck Norris ne sait pas à quoi ressemble Nicolas Sarkozy,  en effet Chuck Norris ne baisse jamais les yeux.': '8.40', "Chuck Norris n'a pas de père. On ne nique pas la mère de Chuck Norris.": '8.34', '': '8.34', "Chuck Norris ne ment pas, c'est la vérité qui se trompe.": '8.33', 'Les samouraïs tuent des mouches avec leurs sabres...Chuck Norris, lui, tue des samouraïs avec des mouches': '8.32', "Les ennemis de Chuck Norris lui disent souvent d'aller au diable. Le Diable aimerait bien qu'ils arrêtent.": '8.30', "Cherchez l'intrus : Un rouge-gorge, un pigeon, un moineau et Chuck Norris.Réponse : Un rouge-gorge, un pigeon et un moineau.": '8.30', 'Chuck Norr

#### Creation d'un dataframe correspondant

In [57]:
Chuck_Norris_jokes = pd.DataFrame(jokes_dict.items(), columns=['Joke', 'Rating'])

In [58]:
pd.set_option('display.max_colwidth', 1000)
Chuck_Norris_jokes.head()

,Joke,Rating
0,Chuck Norris peut manger plus de donuts qu'Homer.,8.87
1,Les ennemis des amis de Chuck Norris sont ses amis. Et oui! Les ennemis de Chuck Norris n'existent plus.,8.83
2,"Un jour Chuck Norris a eu un zero en latin, depuis c'est une langue morte.",8.42
3,L'avenir se demande parfois ce que Chuck Norris lui réserve.,8.42
4,"Chuck Norris ne sait pas à quoi ressemble Nicolas Sarkozy, en effet Chuck Norris ne baisse jamais les yeux.",8.40
